In [38]:
import numpy as np
import scipy.optimize as optim
import matplotlib.pyplot as plt
pi = np.pi

## Task 1 First Pass

In [49]:
# Define collected data
carriage_offset_mm = 80.0 # from waveguide end to start of carriage
mm_to_m = 1000

task11_nearest_minimum_mm = 11.85
task11_sigma_nearest_minimum_mm = 0.02
task11_farthest_minimum_mm = 86.45
task11_sigma_farthest_minimum_mm = 0.02
task11_num_intervals = 4
task11_lambda_g_mm = 2 * (task11_farthest_minimum_mm - task11_nearest_minimum_mm) / task11_num_intervals
# Correction to uwave-formulas.jpg: divide variance by N**2; factor out (2/N)
task11_sigma_lambda_g_mm = (2/task11_num_intervals) * np.sqrt(task11_sigma_nearest_minimum_mm**2 + task11_sigma_farthest_minimum_mm**2)

# The Task 1-2 minima are much harder to locate, because they look locally flat.
task12_minima_mm = [0.99, 14.99, 28.03, 42.5, 55.5, 71.5, 86.5]
task12_sigma_minima_mm = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
task12_num_intervals = len(task12_minima_mm)-1
task12_lambda_0_mm = 2 * (task12_minima_mm[-1] - task12_minima_mm[0]) / task12_num_intervals
task12_sigma_lambda_0_mm = (2/task12_num_intervals) * np.sqrt(task12_sigma_minima_mm[-1]**2 + task12_sigma_minima_mm[0]**2)

star_electrical = 1/task12_lambda_0_mm**2 - 1/task11_lambda_g_mm**2
sigma_star_electrical = np.sqrt((4/task12_lambda_0_mm**6) * task12_sigma_lambda_0_mm**2 + (4/task11_lambda_g_mm**2) * task11_sigma_lambda_g_mm**2)
lambda_c_electrical_mm = 1/np.sqrt(star_electrical)
sigma_lambda_c_electrical_mm = np.sqrt(sigma_star_electrical**2 / (4 * star_electrical**2))

task13_a_mm = 22.80
task13_sigma_a_mm = 0.01

star_physical = 1/(2*task13_a_mm)**2
sigma_star_physical = (0.5 / task13_a_mm**3) * task13_sigma_a_mm
lambda_c_physical_mm = 1/np.sqrt(star_physical)
sigma_lambda_c_physical_mm = np.sqrt(0.25 * sigma_star_physical**2 / star_physical**2)


In [41]:
np.sqrt(1/star_electrical)

np.float64(44.1897422969819)

In [42]:
### Print the final values
print("TASK 1 ELECTRICAL MEASUREMENTS")
print(f"Measured guide wavelength = {task11_lambda_g_mm:.2f}±{task11_sigma_lambda_g_mm:.2f} mm")
print(f"Measured free space wavelength = {task12_lambda_0_mm:.2f}±{task12_sigma_lambda_0_mm:.2f} mm")
print(f"Waveguide critical wavelength for TE10 mode = {lambda_c_electrical_mm:.2f}±{sigma_lambda_c_electrical_mm:.2f} mm")


TASK 1 ELECTRICAL MEASUREMENTS
Measured guide wavelength = 37.30±0.01 mm
Measured free space wavelength = 28.50±0.24 mm
Waveguide critical wavelength for TE10 mode = 44.19±0.74 mm


In [43]:
print("TASK 1 CONSISTENCY CHECKS")
# Waveguide measurement
lambda_g_wavemeter_mm = 37.70
frequency_wavemeter = 10.6E9
c = 299792458
lambda_0_wavemeter_mm = 1000 * c/frequency_wavemeter
# Guide wavelength
print(f'{lambda_g_wavemeter_mm}')
# Free space wavelength
print(f'{lambda_0_wavemeter_mm}')
# Critifcal wavelength
print(f'{lambda_c_physical_mm}')

TASK 1 CONSISTENCY CHECKS
37.7
28.282307358490566
45.6


Conclusion: The precision which I attached to my Task 1-1 guide wavelengths is too great. Furthermore, I'm confident that the systemic error was not a mistake as I was able to replicate these measurements exactly today. Remeasuring with the same procedure will not help.

In [70]:
def calc_gamma(vswr, phase_radians):
    mod_gamma = (vswr-1)/(vswr+1)
    gamma = mod_gamma * np.exp(1j * phase_radians)
    return gamma

In [106]:
# Task 2-1 Silvered Plate Termination
task21_nearest_minimum_mm = 11.85 + carriage_offset_mm # same as Task 1-1
task21_sigma_nearest_minimum_mm = 0.02 # same as Task 1-1

task21_min_value_mv = 0.13
task21_max_value_mv = 311.3

task21_vswr = task21_max_value_mv / task21_min_value_mv
task21_phase = pi + 4*pi*task21_nearest_minimum_mm / task11_lambda_g_mm
task21_phase = task21_phase % 2*pi
task21_phase = 2*(task21_phase % pi) - task21_phase

task21_gamma = calc_gamma(task21_vswr, task21_phase)
print(task21_gamma) # inductive

(0.9630370010661875+0.26625310224458j)


In [107]:
# Task 2-2 Free Space Termination
task22_nearest_min_mm = 17.00 + carriage_offset_mm
task22_sigma_nearest_min_mm = 0.1
task22_min_value_mv = 149.9
task22_sigma_min_value_mv = 0.2
task22_max_value_mv = 205.8
task22_sigma_max_value_mv = 0.2

task22_vswr = task22_max_value_mv / task22_min_value_mv
task22_phase = pi + 4*pi*task22_nearest_min_mm / task11_lambda_g_mm
task22_phase = task22_phase % (2*pi)
task22_phase = 2*(task22_phase % pi) - task22_phase
task22_gamma = calc_gamma(task22_vswr, task22_phase)
task22_normalized_termination_impedance = (1 + task22_gamma) / (1 - task22_gamma)
print(task22_gamma)
print(task22_normalized_termination_impedance)

(-0.04755535879183271-0.1497870230471231j)
(0.8709546662162188-0.26752259664049j)


In [115]:
np.absolute(task22_gamma)

np.float64(0.15715490581951086)

In [108]:
# Double-check against smith chart
task22_fractional_wavelengths = task22_nearest_min_mm / task11_lambda_g_mm
task22_phase_deg = np.rad2deg(task22_phase)
print(task22_vswr)
print(task22_fractional_wavelengths)
print(task22_phase_deg)

1.3729152768512343
2.60053619302949
-107.61394101876706


In [ ]:
# Correction to Uwave-scratch-2-2.jpg: the roles of z_0 and z_g should be switched
# (Because in my case, the load is free space, the guide impedance is the reference)
z_free = 376.730313412
z_g = z_free / task22_normalized_termination_impedance
print(z_g)

(395.5401716456879+121.494192406118j)


Instruction: start the ruler at the horizontal axis to the left of the origin. Move 0.1 fractional wavelengths towards the load. Read the impedance on the ruler at the radius matching VSWR (MAX/MIN) = 1.37. The result is a normalized impedance of 0.875, -0.25 (a bit more)

angle of reflection coefficient is -108 degrees

